# 01. 探索的データ分析 (EDA)

**目的**: 「どのような人が生き残る可能性が高かったのか？」を、モデルを作る前にデータから読み解く。

ここで得た仮説 (女性・子供・上流階級が有利) が、後続の特徴量設計 (`src/feature_engineering.py`) の土台になる。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
print('train:', train.shape, '/ test:', test.shape)
train.head()

## 欠損値の確認
Age・Cabin・Embarked に欠損。Cabin は 7 割超が欠損 → そのまま使わず「記録の有無」フラグに変換する方針。

In [ ]:
miss = train.isnull().sum()
print(miss[miss > 0])
print('\n全体生存率:', round(train['Survived'].mean(), 3))

## 属性別の生存率
史実の「女性・子供優先」「階級格差」がデータに現れているかを確認する。

In [ ]:
print('性別:')
print(train.groupby('Sex')['Survived'].mean())
print('\n客室クラス:')
print(train.groupby('Pclass')['Survived'].mean())
print('\n乗船港:')
print(train.groupby('Embarked')['Survived'].mean())

In [ ]:
# 家族サイズと生存率 (U字型: 単身と大家族が不利)
fs = train['SibSp'] + train['Parch'] + 1
surv_by_fs = train.groupby(fs)['Survived'].mean()
surv_by_fs.plot(kind='bar', title='Survival rate by family size')
plt.xlabel('family_size'); plt.ylabel('survival rate'); plt.tight_layout(); plt.show()
print(surv_by_fs)

## EDA から得た仮説 (→ 特徴量設計へ)

1. **性別**: 女性 74% vs 男性 19% → 最重要。
2. **客室クラス**: 1等 63% > 2等 47% > 3等 24% → 経済力・社会的地位が効く。`Fare` も活かすべき。
3. **家族サイズ**: 2〜4人が最も生存。単身・大家族は不利 → `family_size` / `IsAlone`。
4. **名前の敬称 (Title)**: Mr/Mrs/Miss/Master に年齢・性別・地位が凝縮 → 特徴量化。
5. **相互作用**: 「上流階級の女性」のように、組み合わせで生存率が跳ねる → 交互作用項。

これらを `02_v1_baseline.ipynb` (人間の設計) と `03_v2_optimization.ipynb` (自動最適化) で検証する。